In [1]:
!pip install -q \
transformers\
datasets\
accelerate\
bitsandbytes\
peft\
trl\
unsloth\
mlflow\
tqdm\
numpy\
scipy\
sentencepiece\
safetensors\
tensorboard\
wandb\
rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#let's define our base model name and dataset in this section
#also let's try login in to huggin face
#this indicates the model name and dataset name
from huggingface_hub import login
from google.colab import userdata
model_name="Qwen/Qwen2-1.5B-Instruct"
dataset_name="KisanVaani/agriculture-qa-english-only"
access_token=userdata.get("HF_TOKEN")
login(access_token)

In [4]:
# check if cuda is available
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [5]:
import os
adapter_path="/content/drive/MyDrive/agri_assistant/qwen_agriculture_lora/adapter"
print(os.listdir(adapter_path))

['README.md', 'adapter_config.json', 'adapter_model.safetensors']


In [6]:
import json

with open(f"{adapter_path}/adapter_config.json", "r") as f:
    adapter_config = json.load(f)

print(json.dumps(adapter_config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "Qwen/Qwen2-1.5B-Instruct",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "peft_version": "0.19.1",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "q_proj",
    "o_proj",
    "up_proj",
    "down_proj",
    "v_proj",
    "k_proj",
    "gate_proj"
  ],
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trainable_token_in

In [7]:
from dataclasses import dataclass
from typing import Any, Dict, Optional
@dataclass(slots=True)
class Config:
  model_name=model_name
  quantization_config: Optional[Dict[str, Any]] = None
  trust_remote_code:bool=True
  torch_dtype: str = "bfloat16"
  task_type: str = "CAUSAL_LM"

In [10]:
import torch
try:
    from peft import PeftModel # Changed from get_peft_model to PeftModel
except ImportError:  # pragma: no cover - exercised when PEFT is not installed
    PeftModel = None # Assign None if not found

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
except ImportError:  # pragma: no cover - exercised when Transformers is not installed
    AutoModelForCausalLM = None
    AutoTokenizer = None
    BitsAndBytesConfig = None

class ModelLoader:
    """Load a base model, tokenizer, and attach trained adapters."""

    def __init__(self, config: Config) -> None:
        self.config = config

    def _build_quantization_config(self) -> Any:
        """Build a 4-bit quantization configuration ."""
        if BitsAndBytesConfig is None:
            raise RuntimeError("Install Transformers with BitsAndBytes support to enable 4-bit quantization.")
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="bfloat16",
            bnb_4bit_use_double_quant=True,
        )

    def _resolve_torch_dtype(self) -> Any:
        """Resolve the configured torch dtype safely."""
        if torch is None:
            raise RuntimeError("PyTorch is required to load the QLoRA model.")
        return getattr(torch, self.config.torch_dtype, None)

    def load_tokenizer(self) -> Any:
        """Load the tokenizer for the selected base model."""
        if AutoTokenizer is None:
            raise RuntimeError("Transformers is required to load the tokenizer.")
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_name,
            trust_remote_code=self.config.trust_remote_code,
            use_fast=True,
        )
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        return tokenizer

    def models(self) -> Any:
        """Load the base model with 4-bit quantization then attach the adapter ."""
        if AutoModelForCausalLM is None or PeftModel is None:
            raise RuntimeError("Install Transformers, PEFT, and PyTorch to load the base_model.")

        quantization_config = self.config.quantization_config or self._build_quantization_config()
        torch_dtype = self._resolve_torch_dtype()
        # this loads model in 4bits quantization
        base_model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=self.config.trust_remote_code,
            torch_dtype=torch_dtype,
            attn_implementation="sdpa",
        )
        # now let attach adapter path to the base model
        adapter_model = PeftModel.from_pretrained(base_model, adapter_path)

        return base_model, adapter_model


    def load(self) -> tuple[Any, Any]:
        """Load both the tokenizer and the QLoRA-enabled model."""
        tokenizer = self.load_tokenizer()
        base_model,adapter_model = self.models()
        return base_model,adapter_model, tokenizer

In [11]:
# let's load our base_model,adapter model and tokenizer here
config = Config()
loader = ModelLoader(config)
base_model,adapter_model, tokenizer = loader.load()


In [15]:
# # now let's verify if everything is perfectly loaded
# print(base_model)
# print(adapter_model)
# print(tokenizer)
# # print("Base model paramters:",base_model.print_trainable_parameters())
# print("Adapter model parameters:",adapter_model.print_trainable_parameters())

In [14]:
# make adapter model into eval mode
adapter_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

THIS SECTION OF THE CODE IS FOR INFERENCING

In [19]:
def generate_answer(
    question,
    model,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
):
    messages = [
        {
            "role": "user",
            "content": question,
        }
    ]

    # Convert the conversation into Qwen's expected format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Convert text to tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    # Generate the answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Only decode the newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return answer.strip()

In [20]:
# let's test with an example
question = "What are the common causes of yellow leaves in maize?"

answer = generate_answer(question=question,model=adapter_model)

print("Question:")
print(question)

print("\nModel Answer:")
print(answer)

Question:
What are the common causes of yellow leaves in maize?

Model Answer:
Yellowing leaves can be caused by several factors, including nutrient deficiencies, soil pH, and excessive watering or drought. Yellowing can also be caused by diseases such as powdery mildew or bacterial blight. Additionally, pests like aphids and caterpillars can cause damage to the plant's leaves. It is important to monitor the growth of your maize plants regularly and take appropriate measures to prevent any negative effects on the health of the plant.


In [22]:
#now let's test for batch of questions
test_questions = [
    "What are the common causes of yellow leaves in maize?",
    "How can farmers control weeds in maize farms?",
    "What are the symptoms of nitrogen deficiency in crops?",
    "How often should tomatoes be watered?",
    "What are common causes of poor crop growth?",
]
for question in test_questions:

    answer = generate_answer(question=question,model=adapter_model)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

QUESTION:
What are the common causes of yellow leaves in maize?

ANSWER:
Yellowing leaves in maize can be caused by various factors, including:

  * Nutrient deficiencies: Lack of essential nutrients such as nitrogen, potassium, and phosphorus can cause yellow leaves.
  * Environmental stress: High temperatures, drought, and extreme humidity can also affect maize growth and development, resulting in yellow leaves.
  * Infections or diseases: Certain plant diseases or infections can also cause yellowing leaves in maize.
  * Poor management practices: Poor irrigation, overwatering, poor fertilization, and improper planting practices can all contribute to yellowing leaves.

It's important to note that these symptoms may vary depending on the severity of the issue, the location of the plants, and other environmental factors. If you suspect your maize has yellow leaves, it's best to consult with an agricultural expert for proper diagnosis and treatment.
QUESTION:
How can farmers control wee

In [23]:
# now let's perform evaluation on a our base model and then we compare with our finetuned model
# let's test with an example
question = "What are the common causes of yellow leaves in maize?"

answer = generate_answer(question=question,model=base_model)

print("Question:")
print(question)

print("\nModel Answer:")
print(answer)

Question:
What are the common causes of yellow leaves in maize?

Model Answer:
Yellowing of maize leaves is a common symptom of several diseases, including but not limited to: 

1. Maize rust disease (Meloidogyne incognita) 
2. Fusarium wilt (Fusarium graminearum) 
3. Stem rot (Rhizoctonia solani)
4. Pythium root rot (Pythium italicum)

These diseases can cause yellowing or discolored leaves on the plant. It's important to note that these symptoms may vary from one case to another depending on the severity and location of the infection.

Additionally, other factors such as soil acidity, poor drainage, high humidity, and low soil fertility can also affect the health of maize plants and result in yellowing leaves.


In [24]:
#now let's test for batch of questions
test_questions = [
    "What are the common causes of yellow leaves in maize?",
    "How can farmers control weeds in maize farms?",
    "What are the symptoms of nitrogen deficiency in crops?",
    "How often should tomatoes be watered?",
    "What are common causes of poor crop growth?",
]
for question in test_questions:

    answer = generate_answer(question=question,model=base_model)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

QUESTION:
What are the common causes of yellow leaves in maize?

ANSWER:
Yellowing of maize leaves is caused by several factors, including excessive rainfall, low soil moisture, high temperatures, and pests or diseases. If you suspect your maize plants are suffering from yellowing, it's important to take action quickly to prevent further damage and ensure healthy growth.

Excessive rainfall can cause roots to become waterlogged, which can lead to root rot and other issues that can affect the health of your maize plants. Low soil moisture can also lead to nutrient deficiencies, which can cause leaves to turn yellow.

High temperatures can also cause leaf stress, leading to yellowing. Pests such as aphids, caterpillars, and mites can also attack maize leaves and cause them to turn yellow.

To prevent yellowing, it's important to maintain optimal soil moisture levels, control weeds and pest infestations, and provide adequate nutrition for your maize plants. Regular watering and fertilizat

THIS SECTION OF OUR CODE WILL FOCUS ENTIRELY ON EVALUATION OF OUR FINETUNED MODEL.WE WILL BE USING THREE LEVEL OF EVALUATION:

Validation loss,
Perplexity,
Qualitative comparison on unseen questions

NOTE:LOWER PERPLEXITY INDICATES BETTER PREDICTION OF EVALUATION TEXT

Think of perplexity as a "how surprised is the model?" score.

When you fine-tune a language model, perplexity measures how well the model can predict the next word/token in text it hasn't seen before.


In [25]:
# let's load our dataset
from datasets import load_dataset

dataset = load_dataset(
    dataset_name
)

# Shuffle the complete dataset
dataset = dataset["train"].shuffle(seed=42)

# Use 8,000 examples for training
train_dataset = dataset.select(range(8000))

# Use a completely separate 1,000 examples for evaluation
eval_dataset = dataset.select(range(8000, 9000))

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.97M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22615 [00:00<?, ? examples/s]

Training examples: 8000
Evaluation examples: 1000


In [ ]:
from utils.formatting import formatting_function

def notebook_formatting_function(example):
    return formatting_function(example, tokenizer)

In [ ]:
formatted_eval_dataset = eval_dataset.map(
    lambda example: {"text": notebook_formatting_function(example)}
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [28]:
print(formatted_eval_dataset[0]["text"])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Does most of the money I pay for food go back to the farmer?<|im_end|>
<|im_start|>assistant
Not necessarily<|im_end|>



In [29]:
# THIS CODE BLOCK IS FOR CALCULATING OUR EVALUATION METRICS
import math
import torch
from torch.utils.data import DataLoader

def evaluate_perplexity(
    model,
    tokenizer,
    dataset,
    batch_size=2,
    max_length=512,
):
    model.eval()

    losses = []

    for start in range(0, len(dataset), batch_size):

        batch = dataset[start:start + batch_size]

        texts = batch["text"]

        encoded = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        encoded = {
            key: value.to(model.device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            outputs = model(
                **encoded,
                labels=encoded["input_ids"],
            )

        loss = outputs.loss

        if torch.isfinite(loss):
            losses.append(loss.item())

    average_loss = sum(losses) / len(losses)

    perplexity = math.exp(average_loss)

    return {
        "loss": average_loss,
        "perplexity": perplexity,
    }

In [30]:
results = evaluate_perplexity(
    model=adapter_model,
    tokenizer=tokenizer,
    dataset=formatted_eval_dataset,
    batch_size=2,
    max_length=512,
)

print("Evaluation Loss:", results["loss"])
print("Perplexity:", results["perplexity"])

Evaluation Loss: 3.7856665229797364
Perplexity: 44.065031128765774


In [31]:
# let's calculate metrics on our base model
results = evaluate_perplexity(
    model=base_model,
    tokenizer=tokenizer,
    dataset=formatted_eval_dataset,
    batch_size=2,
    max_length=512,
)

print("Evaluation Loss:", results["loss"])
print("Perplexity:", results["perplexity"])

Evaluation Loss: 3.7856665229797364
Perplexity: 44.065031128765774


In [32]:
# let's check if peft is attached to our model
print(type(adapter_model))
print(adapter_model.peft_config)

<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='Qwen/Qwen2-1.5B-Instruct', revision=None, inference_mode=True, r=16, target_modules={'k_proj', 'q_proj', 'gate_proj', 'up_proj', 'down_proj', 'v_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None

In [33]:
print(adapter_model.active_adapter)

default
